# Detecção e Segmentação de Objetos

Neste notebook, iremos explorar o modelo YOLO para detecção e segmentação de objetos.

In [ ]:
from IPython import get_ipython
if 'google.colab' in str(get_ipython()):
    print("Preparando ambiente Google Colab")
    !pip install opencv-python==5.0.0.93 
    !pip install opencv-contrib-python==5.0.0.93
    !pip install ultralytics
    !git clone https://github.com/pvoloshyn/curso-visao-computacional.git
    %cd curso-visao-computacional
else:
    pass

## Carregando bibliotecas

Além das bibliotecas que usamos em outros notebooks, vamos utilizar mais algumas.
* `ultralytics`: Biblioteca que facilita o acesso e uso dos modelos da família YOLO.

In [ ]:
from ultralytics import YOLO, YOLOE, SAM

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
# Indica ao notebook to render figures in-page.
%matplotlib inline  
from IPython.display import Image

## 1. Detecção de Objetos

Para a deteção de objetos, vamos usar a versão 26 do YOLO. Essa versão é bem recente e performa muito bem.

### 1.1. Carregando o modelo

A biblioteca `ultralytics` facilita o acesso ao modelo, baixando e carregando ele.

In [ ]:
# Caso tenha problemas em usar a versão 26, descomente a linha de baixo e comente a outra
#detect_model = YOLO('modelos/yolov8n.pt')
detect_model = YOLO('modelos/yolo26n.pt')

### 1.2. Carregando imagem

In [ ]:
img = cv2.imread('imagens/02/futebol.webp')
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 8))
plt.imshow(img_rgb)
plt.axis('off')
plt.show()

### 1.3. Inferência do Modelo

Vamos detectar as objetos da imagem. Nesse caso, não precisamos nos preocupar em converter a imagem em blob. A biblioteca `ultralytics` facilita isso. A única coisa que precisamos ter atenção é de usarmos imagens no formato RGB.

In [ ]:
results = detect_model(img_rgb)

A variável `results` é uma lista de objetos `Results`, um para cada imagem processada.

### 1.4. Apresentando e Interpretando Resultados

Em modelos de detecção de objetos, usaremos basicamente a propriedade `boxes` do objeto `Results` da imagem.

In [ ]:
for box in results[0].boxes:
    classe = int(box.cls[0])
    confianca = float(box.conf[0])

    print(
        results[0].names[classe],
        round(confianca, 2),
        box.xyxy[0].round().tolist()
    )

Outra coisa que a biblioteca `ultralytics` facilita é a apresentação do resultado com o método `plot()`.

In [ ]:
result_img = results[0].plot()

plt.figure(figsize=(12, 8))
plt.imshow(result_img)
plt.axis('off')
plt.show()

## 2. Segmentação de Objetos

Vamos usar a versão 26-SEG do YOLO. Ela é bastante atual e performa muito bem.

### 2.1. Carregando o modelo

In [ ]:
seg_model = YOLO("modelos/yolo26n-seg.pt")

### 2.2. Inferência do Modelo

Vamos usar a mesma imagem para facilitar a comparação.

In [ ]:
results = seg_model(img_rgb)
result = results[0]

### 2.3. Apresentando e Interpretando Resultados

Agora `Results` traz também a propriedade `masks` com a máscara de segmentação de cada objeto identificado.

In [ ]:

mask = result.masks.data[0].cpu().numpy()

plt.imshow(mask, cmap='gray')
plt.title("Máscara do Objeto 1")
plt.axis('off')
plt.show()


#### 2.3.1. Sobrepondo máscaras

In [ ]:
def apresentar_segmentacoes(img: np.ndarray, result) -> None:
    CORES = [
        (255,   0,   0),   # vermelho
        (  0, 255,   0),   # verde
        (  0,   0, 255),   # azul
        (255, 128,   0),   # laranja
        (255,   0, 255),   # magenta
        (  0, 255, 255),   # ciano
        (255, 255,   0),   # amarelo
        (128,   0, 255),   # roxo
        (255,   0, 128),   # rosa
        (  0, 128, 255),   # azul claro
        (128, 255,   0),   # verde limão
        (255,  64,  64),   # vermelho claro
    ]

    img_copy = img.copy()
    overlay = img.copy()

    for i, (box, mask) in enumerate(zip(result.boxes, result.masks.data)):
        # Escolhe cor aleatória
        cor = CORES[i % len(CORES)]

        # Prepara máscara
        mask = mask.cpu().numpy()
        mask = cv2.resize(
            mask,
            (img.shape[1], img.shape[0]),
            interpolation=cv2.INTER_NEAREST
        )

        mask = mask > 0.5
        overlay[mask] = cor

        # Apresenta classe e grau de confiança
        class_id = int(box.cls[0])
        label = result.names[class_id]
        conf = float(box.conf[0])

        x1, y1, x2, y2 = (
            box.xyxy[0]
            .cpu()
            .numpy()
            .astype(int)
        )

        cv2.rectangle(
            img_copy,
            (x1, y1),
            (x2, y2),
            cor,
            2
        )

        cv2.putText(
            img_copy,
            f"{label} {conf:.2f}",
            (x1, y1-5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            cor,
            2
        )

    # Sobrepõe máscara na imagem
    resultado = cv2.addWeighted(
        overlay,
        0.35,
        img_copy,
        0.65,
        0
    )

    # Apresenta
    plt.figure(figsize=(12, 8))
    plt.imshow(resultado)
    plt.axis('off')
    plt.show()

In [ ]:
apresentar_segmentacoes(img_rgb, result)

## 3. SAM (Segment Anything Model)

A biblioteca `ultralytics` também permite executarmos o modelo SAM, criado pela Meta. 

Ao invés dele se basear em uma lista pré-definida de categorias, ele permite indicarmos o que queremos segmentar através de pontos ou áreas de interesse. A partir disso, ele busca entender o que queremos extrair.

In [ ]:
sam_model = SAM('modelos/mobile_sam.pt')

In [ ]:
img = cv2.imread('imagens/02/luke.jpg')
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 8))
plt.imshow(img_rgb)
plt.show()

Vamos supor que queremos segmentar o cachorro dessa imagem. Basta passarmos a posição para o modelo.

In [ ]:
results = sam_model(img_rgb, points=[[400, 500]])
result = results[0]

In [ ]:
apresentar_segmentacoes(img_rgb, result)

## 4. YOLOE (Real-Time Seeing Anything)

É um modelo de detecção e segmentação de vocabulário aberto (ou YOLO zero-shot).

In [ ]:
yoloe_model = YOLOE('modelos/yoloe-26m-seg.pt')

In [ ]:
img = cv2.imread('imagens/02/iron-maiden.webp')
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 8))
plt.imshow(img_rgb)
plt.axis('off')
plt.show()

Podemos definir o que queremos extrair. No caso, vamos extrair as guitarras (resultados são melhores quando as classes são definidas em inglês).

In [ ]:

yoloe_model.set_classes([
    'eletric guitar',
])

In [ ]:
results = yoloe_model(img_rgb)
result = results[0]

In [ ]:
apresentar_segmentacoes(img_rgb, result)